# TheLook E-commerce 데이터 전처리

## 프로젝트 개요
BigQuery 공개 데이터셋인 **TheLook E-commerce**를 활용한 데이터 전처리 노트북입니다.

이 노트북의 목적은 분석에 바로 사용할 수 있는 **정제된 데이터셋 2개**를 만드는 것입니다.
- `orders.csv` : 유저 정보 + 주문 정보 + 파생 변수 (코호트 분석, LTV 계산용)
- `session_log.csv` : 세션 단위 퍼널 로그 (전환율 분석용)

In [1]:
# 환경 설정
from google.colab import auth, drive
from google.cloud import bigquery
import pandas as pd

auth.authenticate_user()
project_id = 'project'
client = bigquery.Client(project=project_id)
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 데이터 불러오기

BigQuery의 TheLook 공개 데이터셋에서 분석에 필요한 4개 테이블을 로드합니다.

- `users` : 유저 속성 (나이, 성별, 국가, 유입 채널, 가입일)
- `order_items` : 주문 아이템 단위 거래 내역
- `products` : 상품 메타 정보 (카테고리, 브랜드)
- `events` : 세션 단위 행동 로그

**필터 조건** : 데이터 볼륨과 분석 목적을 고려해 2025-01-01 이후 데이터만 사용합니다.

In [2]:
# SQL 쿼리
query_users = """
SELECT id AS user_id, age, gender, country, traffic_source, created_at AS signup_date
FROM `bigquery-public-data.thelook_ecommerce.users`
WHERE created_at >= '2025-01-01'
"""

query_order_items = """
SELECT created_at AS order_date, user_id, order_id, product_id, status, sale_price
FROM `bigquery-public-data.thelook_ecommerce.order_items`
WHERE created_at >= '2025-01-01'
"""

query_products = """
SELECT id AS product_id, department, category, brand
FROM `bigquery-public-data.thelook_ecommerce.products`
"""

query_events = """
SELECT created_at AS event_time, user_id, session_id, event_type, sequence_number, browser, traffic_source
FROM `bigquery-public-data.thelook_ecommerce.events`
WHERE created_at >= '2025-01-01'
"""

# 데이터 로드
df_users = client.query(query_users).to_dataframe()
df_order_items = client.query(query_order_items).to_dataframe()
df_products = client.query(query_products).to_dataframe()
df_events = client.query(query_events).to_dataframe()

# 주문 테이블

유저-주문-상품 3개 테이블을 JOIN하여 분석용 주문 테이블을 구성합니다.  
이후 데이터 검증 및 이상치 제거 → 파생 변수 생성 순서로 진행합니다.

## 테이블 JOIN

`users`를 기준으로 `order_items`를 **LEFT JOIN** 합니다.
구매 이력이 없는 유저도 분석 대상에 포함시키기 위해 LEFT JOIN을 사용합니다.
이후 `products`를 추가로 LEFT JOIN하여 상품 카테고리 정보를 결합합니다.


In [3]:
df_orders = df_users.merge(df_order_items, on='user_id', how='left')\
                    .merge(df_products, on='product_id', how='left')
df_orders_backup = df_orders.copy()
df_orders.head()

,user_id,age,gender,country,traffic_source,signup_date,order_date,order_id,product_id,status,sale_price,department,category,brand
0,70259,12,F,Brasil,Email,2026-01-30 04:38:00+00:00,NaT,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
1,57832,12,F,Brasil,Organic,2026-01-03 14:13:00+00:00,2026-02-01 07:23:24+00:00,72542,3450,Complete,47.880001,Women,Dresses,Evolution by Cyrus
2,57832,12,F,Brasil,Organic,2026-01-03 14:13:00+00:00,2026-02-01 08:20:00+00:00,72542,5368,Complete,99.000000,Women,Pants & Capris,Jones New York
3,41683,12,F,Brasil,Search,2025-09-24 05:04:00+00:00,NaT,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
4,62319,12,M,Brasil,Search,2025-05-02 07:52:00+00:00,2025-07-02 20:26:50+00:00,78084,16591,Cancelled,28.950001,Men,Tops & Tees,Harley-Davidson


## 데이터 검증 및 전처리



In [4]:
# 타임존 제거
df_orders['order_date'] = pd.to_datetime(df_orders['order_date']).dt.tz_localize(None)
df_orders['signup_date'] = pd.to_datetime(df_orders['signup_date']).dt.tz_localize(None)

# [Data Check]
# 유저 정보 일관성 확인
attr_err = (df_orders.groupby('user_id')[['age', 'country', 'gender']].nunique() > 1).any(axis=1).sum()

# 비정상 가격 확인
price_err = (df_orders['sale_price'] <= 0).sum()

# 가입 전 주문 확인
timeline_err = (df_orders['order_date'].dt.date < df_orders['signup_date'].dt.date).sum()

# 동일 주문 내 날짜 불일치 확인
orderdate_err = (df_orders.groupby('order_id')['order_date'].nunique() > 1).sum()

print(f"--- Check Result ---")
print(f"1. 유저 속성 불일치: {attr_err} 건")
print(f"2. 비정상 가격     : {price_err} 건")
print(f"3. 가입 전 주문    : {timeline_err} 건")
print(f"4. 주문 날짜 불일치: {orderdate_err} 건")

# [Data Cleaning]
# 가입 전 주문 제거
df_orders = df_orders[~(df_orders['order_date'].notnull() & (df_orders['order_date'].dt.date < df_orders['signup_date'].dt.date))].copy()

# 동일 주문(order_id) 내 날짜 불일치 건은 최솟값으로 보정
df_orders['order_date'] = df_orders.groupby('order_id')['order_date'].transform('min')

--- Check Result ---
1. 유저 속성 불일치: 0 건
2. 비정상 가격     : 0 건
3. 가입 전 주문    : 6 건
4. 주문 날짜 불일치: 7621 건


### 전처리 결과 확인

In [5]:
# 가입 전 주문 확인
timeline_err = (df_orders['order_date'].dt.date < df_orders['signup_date'].dt.date).sum()

# 동일 주문 내 날짜 불일치 확인
orderdate_err = (df_orders.groupby('order_id')['order_date'].nunique() > 1).sum()

print(f"가입 전 주문    : {timeline_err} 건")
print(f"주문 날짜 불일치: {orderdate_err} 건")

가입 전 주문    : 0 건
주문 날짜 불일치: 0 건


## 파생 변수 생성

코호트 분석과 LTV 계산에 필요한 변수를 추가합니다.

- `first_order_date` : 유저별 첫 구매일 → 리텐션 코호트 기준
- `diff_month_from_first_order` : 첫 구매 이후 경과 월 → 코호트 열(column) 인덱스
- `diff_month_from_signup` : 가입 이후 경과 월 → LTV 계산의 시간축

In [6]:
# 유저별 첫 구매일
df_orders['first_order_date'] = df_orders.groupby('user_id')['order_date'].transform('min')

In [7]:
# 첫 구매 이후 경과 월
df_orders['diff_month_first_order'] = (
    df_orders['order_date'].dt.to_period('M') -
    df_orders['first_order_date'].dt.to_period('M')
).apply(lambda x: x.n if pd.notna(x) else pd.NA).astype('Int64')

In [8]:
# 가입 후 경과 월
df_orders['diff_month_signup'] = (
    df_orders['order_date'].dt.to_period('M') -
    df_orders['signup_date'].dt.to_period('M')
).apply(lambda x: x.n if pd.notna(x) else pd.NA).astype('Int64')

In [9]:
# 컬럼 순서 변경
orders_cols = [
    'order_id', 'user_id', 'product_id',
    'status', 'sale_price',
    'department', 'category', 'brand',
    'gender', 'age', 'country', 'traffic_source',
    'signup_date', 'first_order_date', 'order_date',
    'diff_month_signup', 'diff_month_first_order'
]

df_orders = df_orders[orders_cols].copy()

In [10]:
df_orders.head()

,order_id,user_id,product_id,status,sale_price,department,category,brand,gender,age,country,traffic_source,signup_date,first_order_date,order_date,diff_month_signup,diff_month_first_order
0,<NA>,70259,<NA>,NaN,NaN,NaN,NaN,NaN,F,12,Brasil,Email,2026-01-30 04:38:00,NaT,NaT,<NA>,<NA>
1,72542,57832,3450,Complete,47.880001,Women,Dresses,Evolution by Cyrus,F,12,Brasil,Organic,2026-01-03 14:13:00,2026-02-01 07:23:24,2026-02-01 07:23:24,1,0
2,72542,57832,5368,Complete,99.000000,Women,Pants & Capris,Jones New York,F,12,Brasil,Organic,2026-01-03 14:13:00,2026-02-01 07:23:24,2026-02-01 07:23:24,1,0
3,<NA>,41683,<NA>,NaN,NaN,NaN,NaN,NaN,F,12,Brasil,Search,2025-09-24 05:04:00,NaT,NaT,<NA>,<NA>
4,78084,62319,16591,Cancelled,28.950001,Men,Tops & Tees,Harley-Davidson,M,12,Brasil,Search,2025-05-02 07:52:00,2025-07-02 20:26:50,2025-07-02 20:26:50,2,0


## 최종 데이터셋 저장

In [11]:
# csv로 저장
df_orders.to_csv('orders.csv',index=False)

# 세션 로그

이벤트 로그(`df_events`)를 **세션 단위**로 집계하여 퍼널 분석용 테이블을 만듭니다.

**목표 구조** : 1세션 = 1행, 각 퍼널 단계(조회/장바구니/구매) 도달 여부와 소요 시간 포함

In [12]:
df_events.head()

,event_time,user_id,session_id,event_type,sequence_number,browser,traffic_source
0,2025-07-26 15:22:00+00:00,<NA>,3c22e832-ddb1-49ca-9441-43bf1a0faa67,department,1,Firefox,Adwords
1,2025-07-04 17:47:00+00:00,<NA>,1a76ceb5-6eca-4d52-ac67-59bb6a495572,department,1,Safari,Email
2,2026-04-25 13:40:00+00:00,<NA>,2050dc99-ec39-4757-be52-5b54d1ad4f9a,department,1,Other,Adwords
3,2025-06-22 12:42:00+00:00,<NA>,52b3a39a-e0c1-4842-a320-345dfd767e45,department,1,Chrome,Email
4,2025-08-05 02:53:00+00:00,<NA>,72b3b221-f75e-4412-9534-5ebdb96a117e,department,1,Firefox,Facebook


## 데이터 전처리

In [13]:
# 타임존 제거

df_events['event_time'] = pd.to_datetime(df_events['event_time']).dt.tz_localize(None).dt.floor('s')

## 세션 로그 집계

In [14]:
# 세션별 이벤트 시점 피벗팅
session_pivot = df_events.pivot_table(
    index='session_id',
    columns='event_type',
    values='event_time',
    aggfunc='min'
)

# 세션 기본 정보 결합
session_base = df_events.groupby('session_id').agg(
    user_id=('user_id', 'first'),
    traffic_source=('traffic_source', 'first'),
    browser=('browser', 'first'),
    session_start_at=('event_time', 'min'),
    landing_page=('event_type', 'first')
)

# 최종 세션 로그 병합
session_log = session_base.join(session_pivot[['product', 'cart', 'purchase']]).reset_index()

# 소요 시간 계산
session_log['view_to_cart_sec'] = (session_log['cart'] - session_log['product']).dt.total_seconds()
session_log['cart_to_purchase_sec'] = (session_log['purchase'] - session_log['cart']).dt.total_seconds()

# 도달 여부 플래그
session_log['has_landing'] = session_log['session_start_at'].notnull().astype(int)
session_log['has_view'] = session_log['product'].notnull().astype(int)
session_log['has_cart'] = session_log['cart'].notnull().astype(int)
session_log['has_purchase'] = session_log['purchase'].notnull().astype(int)

## 이상치 처리

### 이상치 기준 설정 : 소요 시간

`describe()` 결과를 보면 두 소요 시간 컬럼의 분포가 크게 다릅니다.

- `view_to_cart_sec` : 중앙값 125초, 최대 1,740초(29분) → **정상 범위**
- `cart_to_purchase_sec` : 중앙값 304초이지만 75% 구간이 173,355초(약 48시간), 최대 347,015초(약 4일) → **이상치 다수 존재**

`cart_to_purchase_sec`의 이상치는 단순히 탭을 오랫동안 열어둔 경우처럼 '하나의 세션' 정의를 벗어난 케이스로 판단합니다.
**1시간(3,600초)을 초과하는 세션은 단일 세션으로 보기 어렵다고 판단하여 제거합니다.**

In [15]:
session_log[['view_to_cart_sec', 'cart_to_purchase_sec']].describe()

,view_to_cart_sec,cart_to_purchase_sec
count,136699.000000,91527.000000
mean,347.314252,89631.319534
std,475.175491,123506.108630
min,0.000000,0.000000
25%,60.000000,92.000000
50%,125.000000,302.000000
75%,420.000000,173347.000000
max,1740.000000,347086.000000


In [16]:
# [Data Check]
# Landing이 Purchase인 경우
landing_err = (session_log['landing_page'] == 'purchase').sum()

# 순서 역전 확인
seq_err = ((session_log['product'] > session_log['cart']) | (session_log['cart'] > session_log['purchase'])).sum()

# 소요 시간 1시간(3600초) 초과
timeout_err = (session_log['cart_to_purchase_sec'] > 3600).sum()
print(f"--- Check Result ---")
print(f"1. Landing = Purchase  : {landing_err} 건")
print(f"2. 순서 역전 세션      : {seq_err} 건")
print(f"3. 세션 시간 1시간 초과: {timeout_err} 건")

# [Data Cleaning]
# Landing Page가 Purchase 인 세션 제거
session_log = session_log[session_log['landing_page'] != 'purchase'].copy()

# 소요시간 1시간 초과 세션 제거
session_log = session_log[(session_log['cart_to_purchase_sec'].isna()) | (session_log['cart_to_purchase_sec'] <= 3600)].copy()

--- Check Result ---
1. Landing = Purchase  : 108 건
2. 순서 역전 세션      : 0 건
3. 세션 시간 1시간 초과: 37770 건


### 전처리 결과 확인

In [17]:
# 소요시간 분포 확인
display(session_log['cart_to_purchase_sec'].describe())

landing_err = (session_log['landing_page'] == 'purchase').sum()
timeout_err = (session_log['cart_to_purchase_sec'] > 3600).sum()

print(f"\nLanding = Purchase  : {landing_err} 건")
print(f"세션 시간 1시간 초과: {timeline_err} 건")

,cart_to_purchase_sec
count,53757.000000
mean,171.704894
std,214.295498
min,0.000000
25%,54.000000
50%,109.000000
75%,163.000000
max,1533.000000



Landing = Purchase  : 0 건
세션 시간 1시간 초과: 0 건


## 최종 데이터셋 저장

In [18]:
# 유지할 컬럼만 선택
session_log_cols = [
    'session_id', 'user_id', 'session_start_at',
    'landing_page', 'traffic_source', 'browser',
    'has_landing', 'has_view', 'has_cart', 'has_purchase',
    'view_to_cart_sec', 'cart_to_purchase_sec'
]

df_session_log = session_log[session_log_cols].copy()

# CSV 저장 (인덱스 제외)
df_session_log.to_csv('session_log.csv', index=False)